In [4]:
import re
import emoji
emoji_dict = {
    # Nhóm cười mỉa mai, cợt nhả (Rất quan trọng trong Toxic Comment)
    "😂": " cười_ra_nước_mắt ", 
    "🤣": " cười_lăn_lộn ",
    "😅": " cười_ngượng_ngùng ",
    "😆": " cười_hết_mức ",
    "🤡": " mặt_hề_mỉa_mai ",
    "🙃": " mỉa_mai_ngược ",
    "🙄": " khinh_thường ",
    "🤭": " cười_che_miệng ",

    # Nhóm cảm xúc tích cực (Giúp mô hình nhận diện câu Clean)
    "🥰": " yêu_thương ",
    "❤": " yêu_thích ",
    "😁": " cười_tươi ",
    "😀": " cười_vui ",
    "😌": " nhẹ_nhõm ",
    "🙂": " cười_nhẹ ",
    "🍀": " may_mắn ",

    # Nhóm cảm xúc mạnh/Tiêu cực
    "😭": " khóc_lớn ",
    "😳": " ngạc_nhiên_đỏ_mặt ",
    "🥺": " nũng_nịu ",
    "🤔": " đang_suy_nghĩ ",
    "🥲": " cười_trong_nước_mắt ",
    "😱": " hoảng_sợ ",
    "🤧": " hắt_xì_ốm ",
    
    # Bổ sung các icon Toxic tiềm năng (Dù chưa có trong top 20 nhưng rất hay gặp)
    "😡": " giận_dữ ",
    "🤬": " chửi_thề ",
    "🖕": " ngón_tay_thối ",
    "👎": " phản_đối ",
    "🤢": " kinh_tởm ",
    "💩": " rác_rưởi "
}

emoticon_dict = {
    ":)))": " cười_vui_sướng ",
    ":))": " cười_vui ",
    ":)": " cười_nhẹ_mỉa_mai ",
    "=))": " cười_lăn_lộn ",
    "^^": " hạnh_phúc ",
    ":D": " cười_tươi ",
    ":P": " lè_lưỡi ",
    "(y)": " đồng_ý ", # Biểu tượng like cũ của Facebook
    "<3": " yêu_thương "
}

def clean_text(text):

    
    text = text.lower()

    # TH1: Loại bỏ các ký tự lặp Thay \1 bằng \1\1
    text = re.sub(r'([a-z])\1+', r'\1\1', text)
    # TH2: Mapping các slang word cơ bản
    text = normalize_slang(text)
    # TH3: Loại bỏ url, hashtag
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    # TH4: Mapping các icon quá từ dễ nhận biết hơn
    text = convert_emojis(text)
    text = convert_emoticons(text)
    return text


def convert_emoticons(text):
    # 1. Xử lý các dạng :))) hoặc :)))) (nhiều dấu ngoặc)
    # Quy luật: Tìm dấu : hoặc = theo sau là ít nhất 2 dấu )
    text = re.sub(r'[:=]\)+', ' cười_vui ', text)
    
    # 2. Xử lý dạng cười lăn lộn =))
    text = re.sub(r'=\)+', ' cười_lăn_lộn ', text)
    
    # 3. Xử lý các dạng emoticon cố định khác từ dict
    for emo, text_val in emoticon_dict.items():
        if emo in text:
            text = text.replace(emo, text_val)
            
    return text

def convert_emojis(text):
    processed_text = text
    
    # 1. Dùng vòng lặp để thay thế từ điển thủ công (Dùng 'emj' thay vì 'emoji')
    for emj, text_val in emoji_dict.items():
        processed_text = processed_text.replace(emj, text_val)
    
    # 2. Sau khi thoát vòng lặp, mới dùng thư viện emoji để xử lý nốt
    # Lúc này Python sẽ hiểu 'emoji' ở đây là cái bạn đã 'import' ở trên
    final_text = emoji.demojize(processed_text, delimiters=(" ", " "))
    
    return final_text


def normalize_slang(text):

    
    slang_dict = {
    "zậy": "vậy", "zay": "vậy", "j": "gì", "bit": "biết",
    "dc": "được", "đc": "được", "hok": "không", "ko": "không", "k": "không",
    "đm": "địt mẹ", "dm": "địt mẹ", "vcl": "vãi cả luyện", "vl": "vãi luyện",
    "chít": "chết", "eny": "em người yêu", "ny": "người yêu",
    "trois": "trời", "trùi": "trời", "tr": "trời", "chets":"chết"
    }

    slang_dict.update({
    "j": "gì", "gi": "gì", "jz": "gì vậy", "z": "vậy", 
    "zay": "vậy", "zậy": "vậy", "s": "sao", "ms": "mới",
    "w": "quá", "wen": "quen", " bít": "biết", "mún": "muốn",
    "thui": "thôi", "hẻ": "hả", "r": "rồi", "gùi": "rồi"
})
    
    slang_dict.update({
    # Biến thể của Đ*
    "đm": "địt mẹ", "dm": "địt mẹ", "đmm": "địt mẹ mày", "dmm": "địt mẹ mày",
    "đcm": "địt con mẹ", "dcm": "địt con mẹ", "đcmm": "địt con mẹ mày",
    "clm": "con lồn mẹ", "clme": "con lồn mẹ", "cmn": "con mẹ nó",
    
    # Biến thể của L* và C*
    "cl": "cái lồn", "cailon": "cái lồn", "l": "lồn", "ln": "lồn",
    "cc": "củ các", "củ kẹc": "củ các", "kẹc": "các", "cac": "các",
    
    # Biến thể của S*
    "sml": "sấp mặt lồn", "ml": "mặt lồn", "hãm l": "hãm lồn",
    
    # Biến thể của Chó / Ngu
    "ch0": "chó", "cờ hó": "chó", "choá": "chó",
    "ngu lol": "ngu lồn", "ngu l": "ngu lồn", "óc c": "óc chó",
    
    # Biến thể khác
    "đéo": "không", "đ": "đéo", "éo": "không",
    "vcl": "vãi cả luyện", "vl": "vãi luyện", "vcln": "vãi cả luyện luôn",
    "đỉ": "đĩ", "điên": "đin", "khùng": "khùng"
})
    # Tách từ dựa trên khoảng trắng
    words = text.split()
    # Thay thế nếu từ đó nằm trong từ điển, nếu không giữ nguyên
    res = [slang_dict[w] if w in slang_dict else w for w in words]
    return " ".join(res)




In [3]:
import pandas as pd
import emoji
from collections import Counter

df = pd.read_csv('final_data.csv')

def extract_emojis(text):
    if not isinstance(text, str):
        return []
    # Trích xuất danh sách các emoji từ văn bản
    return [c for c in text if emoji.is_emoji(c)]

# 1. Thu thập tất cả icon vào một list khổng lồ
all_emojis = []
for comment in df['comment']: # Thay 'text' bằng tên cột của bạn
    all_emojis.extend(extract_emojis(comment))

# 2. Đếm tần suất
emoji_counts = Counter(all_emojis)

# 3. Chuyển thành DataFrame để dễ quan sát
emoji_stats = pd.DataFrame(emoji_counts.items(), columns=['Icon', 'Frequency'])
emoji_stats = emoji_stats.sort_values(by='Frequency', ascending=False).reset_index(drop=True)

# 4. Thêm cột ý nghĩa tiếng Anh (để bạn dễ map sang tiếng Việt)
emoji_stats['English_Name'] = emoji_stats['Icon'].apply(lambda x: emoji.demojize(x, delimiters=("", "")))

print(emoji_stats.head(20)) # Xem top 20 icon xuất hiện nhiều nhất

   Icon  Frequency                    English_Name
0     😂        637          face_with_tears_of_joy
1     🤣        403   rolling_on_the_floor_laughing
2     🥰        211        smiling_face_with_hearts
3     😭        142              loudly_crying_face
4     😁        109  beaming_face_with_smiling_eyes
5     😅         88        grinning_face_with_sweat
6     😆         79         grinning_squinting_face
7     🍀         64                four_leaf_clover
8     😳         61                    flushed_face
9     🥺         60                   pleading_face
10    ❤         56                       red_heart
11    😌         56                   relieved_face
12    🤔         51                   thinking_face
13    🥲         48          smiling_face_with_tear
14    😀         46                   grinning_face
15    🤡         36                      clown_face
16    🤭         36       face_with_hand_over_mouth
17    😱         34          face_screaming_in_fear
18    🙂         31           sl

In [ ]:
import pandas as pd
from tqdm import tqdm


df = pd.read_csv('final_data.csv')

tqdm.pandas()

df['clean_segmented_text'] = df['comment'].progress_apply(clean_text)

df.to_csv('final_segment_data.csv', index=False, encoding='utf-8-sig')


100%|██████████| 9155/9155 [00:00<00:00, 19707.23it/s]


In [3]:
import py_vncorenlp
import os
import pandas as pd
from tqdm import tqdm

# --- PHẦN 1: KHỞI TẠO (CHỈ CHẠY 1 LẦN) ---
save_dir = r'D:\vncorenlp'
jar_path = os.path.join(save_dir, 'VnCoreNLP-1.2.jar')

if not os.path.exists(jar_path):
    print("❌ Không tìm thấy file JAR!")
else:
    print("✅ Đang khởi tạo Java VnCoreNLP (Vui lòng đợi)...")
    # Khởi tạo rdrsegmenter ở biến TOÀN CỤC (ngoài hàm)
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=save_dir)

# --- PHẦN 2: HÀM XỬ LÝ ---
def segment_comment(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    try:
        # Sử dụng biến rdrsegmenter đã khởi tạo ở trên
        output = rdrsegmenter.word_segment(text)
        return " ".join(output)
    except Exception as e:
        return text # Trả về text gốc nếu lỗi

# --- PHẦN 3: CHẠY VỚI PANDAS ---
tqdm.pandas()
# Giả sử df đã có sẵn
df['segment_comment'] = df['clean_segmented_text'].progress_apply(segment_comment)

# Lưu ý thêm index=False để file CSV không bị dư cột số thứ tự ở đầu
current_dir = os.getcwd()
file_path = os.path.join(current_dir, "final_segment_data.csv")
df = df[['segment_comment', 'label_id']]
df = df.rename(columns={
    'segment_comment':'comment'
})
df.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"✅ Đã lưu file thành công tại: {file_path}")

✅ Đang khởi tạo Java VnCoreNLP (Vui lòng đợi)...


100%|██████████| 9155/9155 [00:02<00:00, 3369.40it/s]

✅ Đã lưu file thành công tại: D:\vncorenlp\final_segment_data.csv
